<a href="https://colab.research.google.com/github/robotlover2/deeplearning26.08.10/blob/main/34_%EB%AA%A8%EB%8D%B8%EA%B2%BD%EB%9F%89%ED%99%94_Export_%ED%8C%A8%ED%82%A4%EC%A7%95_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Custom 모델 경량화 + RPi5 배포 패키지 (PC)

## 학습 목표
1. `best.pt` → **LiteRT(.tflite)** 4종 변환 : FP32 / 동적INT8 / 정적INT8 / 640
2. **양자화가 정확도를 얼마나 깎는지** 직접 측정 (안전 과제의 핵심)
3. RPi5 배포용 zip 패키지 생성

## 입력 (커스텀 모델 산출물)
- `best.pt`
- `data.yaml` ← INT8 캘리브레이션용
- `valid/images/` ← 캘리브레이션 사진

## 실행 환경
- Colab CPU 무료 등급으로 충분 (GPU 불필요)
- LiteRT(Lite Runtime) 변환은 Linux x86_64 / macOS 에서만 된다 → **라즈베리파이에서는 못 만든다**


---
## 바뀐 점 — 예전 자료를 쓰고 있다면 반드시 확인

**Ultralytics 8.4.83** 부터 `tflite` export 가 **LiteRT** 로 통합됐다.
파일 확장자는 그대로 `.tflite` 이지만, 옵션 이름이 전부 바뀌었다.

| 예전 코드 | 지금 코드 |
| --- | --- |
| `format='tflite'` | `format='litert'` |
| `half=True` (FP16) | **없어졌다** — FP16은 GPU 전용이라 RPi5에 무의미 |
| `int8=True, data=...` | `quantize=8, data=...` |
| — | `quantize='w8a32'` **(새로 생김 — 데이터 없이 되는 INT8)** |
| `tensorflow`, `onnx2tf` 수동 설치 | **불필요** — 버전 지옥이 사라졌다 |

경량화가 훨씬 쉬워짐



---
## 경량화의 세 축 (1분 정리)

| 축 | 무엇을 바꾸나 | 효과 |
| --- | --- | --- |
| **A. 엔진** | PyTorch → LiteRT | 속도 2~3배. **크기·정확도는 그대로** |
| **B. 정밀도** | FP32 → INT8 | 크기 1/4, 속도↑, **정확도 조금↓** |
| **C. 입력** | 640 → 320 | 속도 4배, **작은 물체 놓침** |

> 축 A는 *짐을 줄인 게 아니라 길을 바꾼 것*이다. 진짜 압축은 축 B부터다.
> 앞서 파이에서 돌려 본 `yolov8n_int8.tflite` 가 빨랐던 이유가 바로 축 B다.


---
## [STEP 0] 환경 설치

In [1]:
# 이 셀만 실행한 뒤 → [런타임 → 세션 다시 시작] → STEP 1 부터
!pip install -q -U ultralytics

# LiteRT 변환에 필요한 추가 패키지를 실제로 불러오게 만드는 사전 변환 테스트 - 더미 변환을 한 번 돌려 LiteRT 변환기를 미리 설치
# 변환 잘 되는지 확인만 해보도록 함
!yolo export model=yolo11n.pt format=litert imgsz=320


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.2/53.2 kB 2.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from 'yolo11n.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 84, 2100) (5.4 MB)
requirements: Ultralytics

In [2]:
import os, json, shutil, time # 폴더 만들기 , 파일 이름 바꾸기, 세팅값 제기 ,시간 제기
from pathlib import Path
from ultralytics import YOLO
import ultralytics
print('ultralytics', ultralytics.__version__, '(8.4.83 이상이어야 한다)')

ultralytics 8.4.118 (8.4.83 이상이어야 한다)


---
## [STEP 1] 입력 파일 준비

PART 2-A 의 `safety_train_out.zip` 을 올린다. (강사 배포본은 `gdown` 으로 받는다)

In [6]:
# 방법 ① 강사 배포본
# !gdown 1t9x4_RYVirHhEYsrX2wY9ktEjjYVhCv3
# !unzip -q -o safety_train_out.zip -d safety_train_out

# 방법 ② 직접 업로드 — 위가 안 되면 아래 두 줄의 주석을 푼다
from google.colab import files
files.upload()
!unzip -q -o safety_train_out.zip -d safety_train_out

Saving safety_train_out.zip to safety_train_out.zip


- 모델과 데이터셋을 자동으로 찾아서, 어느 컴퓨터에서 실행해도 오류가 나지 않도록 준비
    ```
    ① best.pt 자동 찾기
          │
          ▼
    ② data.yaml 자동 찾기
          │
          ▼
    ③ data.yaml 경로 수정: 현재 컴퓨터에 맞게 경로 수정
          │
          ▼
    ④ Validation 이미지 확인 : 캘리브레이션은 다시 학습하는 것이 아니라, 숫자를 압축하기 전에 적절한 압축 기준을 찾는 과정
          │
          ▼
    ⑤ YOLO 모델 불러오기
    ```



| 코드                   | 하는 일         | 왜 필요한가?                   |
| -------------------- | ------------ | ------------------------- |
| `rglob('best.pt')`   | 모델 자동 찾기     | 폴더 위치를 몰라도 됨              |
| `rglob('data.yaml')` | 데이터 설정파일 찾기  | 데이터셋 정보 확인                |
| `yaml.safe_load()`   | data.yaml 읽기 | 설정 내용 확인                  |
| `cfg['path'] = ...`  | 데이터셋 경로 수정   | 현재 컴퓨터에 맞게 변경             |
| `resolve()`          | 절대경로 생성      | 경로 오류 방지                  |
| `len(...)`           | 검증 이미지 개수 확인 | INT8 Calibration 가능 여부 확인 |
| `YOLO(best.pt)`      | 모델 불러오기      | 변환 준비 완료                  |


In [7]:
# best.pt 와 data.yaml 을 폴더 어디에 있든 찾아낸다
import yaml
PT_PATH   = next(Path('.').rglob('best.pt'), None)
YAML_PATH = next(Path('.').rglob('data.yaml'), None)
assert PT_PATH, 'best.pt 를 찾지 못했다 — 모델 생성 산출물을 확인한다' # 찾지 못했을 때

# data.yaml 의 path 를 '지금 이 컴퓨터의 절대 경로' 로 고친다 ★
# 만든 곳과 푸는 곳이 다르므로 이 한 단계가 없으면 INT8 변환이 실패한다.
if YAML_PATH:
    cfg = yaml.safe_load(open(YAML_PATH, encoding='utf-8'))
    ds  = (YAML_PATH.parent / cfg.get('path', '.')).resolve() #데이터의 현재 경로를 찾아서 resolve로 절대경로로 만들어 주면됌
    cfg['path'] = str(ds) # 절대 경로 지정
    yaml.dump(cfg, open(YAML_PATH, 'w', encoding='utf-8'),
              sort_keys=False, allow_unicode=True) #INT8로 변환하면 됨
    n = len(list((ds / cfg['val']).glob('*'))) if (ds / cfg['val']).exists() else 0
    print(f'캘리브레이션 사진 : {n} 장  ({ds / cfg["val"]})')
    if n == 0:
        print('  → 사진이 없다. 정적 INT8(STEP 5)은 건너뛰게 된다.')

print('모델      :', PT_PATH)
print('data.yaml :', YAML_PATH or '없음 (정적 INT8·정확도 측정 STEP을 건너뛴다)')

model = YOLO(str(PT_PATH))

캘리브레이션 사진 : 113 장  (/content/safety_train_out/safety_dataset/valid/images)
모델      : safety_train_out/best.pt
data.yaml : safety_train_out/data.yaml


---
## [STEP 2] 클래스 순서 확인 ★

모델은 클래스를 **이름이 아니라 번호로** 내보낸다.
파이 실습에서 `coco_yolo.json` 없이는 결과를 못 읽었던 것과 같은 이야기다.

**이 번호가 D9에서 아두이노로 보낼 신호의 기준이 된다. 순서가 바뀌면 장치가 거꾸로 움직인다.**

In [15]:
CLASSES = [model.names[i] for i in sorted(model.names)]
DANGER  = next((i for i, n in enumerate(CLASSES)
                if n.lower().replace('-', '').replace('_', '').startswith('nohelmet')), 0)

for i, n in enumerate(CLASSES):
    print(f'  {i} : {n}' + ('   ← 위험 신호' if i == DANGER else ''))

  0 : No-helmet   ← 위험 신호
  1 : helmet


---
## [STEP 3] LiteRT FP32 320 — 축 A만 켠다

엔진만 LiteRT 로 바꾼다. 양자화를 하지 않고 실행 형식만 PyTorch → LiteRT로 변경했으므로 정확도는 **원본과 거의 동일한 수준**이어야 한다.

In [8]:
MADE, LOG = {}, {}

def export_litert(tag, imgsz, **kw):#tag=='fp32_320 걍이름이고 imgsz=320 , **kw
    t = time.time()
    out = YOLO(str(PT_PATH)).export(format='litert', imgsz=imgsz, **kw)  # PT로 받은 YOLO모델을 litert로 export 하겠다
    LOG[tag] = time.time() - t
    dst = f'{tag}.tflite'
    shutil.copy(out, dst)
    MADE[tag] = dst
    print(f'{tag:12s} {LOG[tag]:5.0f}s   {os.path.getsize(dst)/1024/1024:.2f} MB')
    return dst

export_litert('fp32_320', 320)  # ③ 라즈베리 파이등의  온디바이스용 파일이 만들어짐

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from 'safety_train_out/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (5.2 MB)



LiteRT: starting export with litert_torch 0.9.3...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

...

(00:18) [ DONE] LiteRT-Torch Convert (+00:18)

(00:00) [START] Write Model to safety_train_out/best.tflite

(00:00) [ DONE] Write Model to safety_train_out/best.tflite (+00:00)

LiteRT: export success ✅ 21.0s, saved as 'safety_train_out/best.tflite' (10.1 MB)

Export complete (21.4s)
Results saved to /content/safety_train_out/best.tflite
Predict:         yolo predict task=detect model=safety_train_out/best.tflite imgsz=320 
Validate:        yolo val task=detect model=safety_train_out/best.tflite imgsz=320 data=/content/helmet--1/data.yaml  
Visualize:       https://netron.app
fp32_320        21s   10.06 MB


'fp32_320.tflite'

---
## [STEP 4] LiteRT 동적 INT8 320 — 축 A + B

`quantize='w8a32'` : 가중치만 INT8, 계산 중간값은 FP32.
**캘리브레이션 데이터가 필요 없고** 정확도 손실이 거의 없다. 가장 안전한 선택.
- w8a32 : 어떤 값을 몇 비트(bit)로 저장하고 계산하는지를 나타내는 표기법
    - w = Weights(가중치)
    - a = Activations(활성값, 중간 계산 결과)
```
w8a32
 │ │
 │ └── Activation : 32bit(FP32)
 │
 └──── Weight : 8bit(INT8)
 ```

In [9]:

# B 크기를 줄여 보려고 함 : 양자화
export_litert('w8a32_320', 320, quantize='w8a32') # ① 학습과정의 가중치는 INT8이지만 중간값은 FP32 로 사용함 즉 과정은 FP32로 결과값만 INT8로 줄임
# 즉  이 과정에서 만들어진 파일을 온디바이스에 올리면 실행이 된다

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from 'safety_train_out/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (5.2 MB)

LiteRT: starting export with litert_torch 0.9.3...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


...

(00:00) [ DONE] Write Model to safety_train_out/best_w8a32.tflite (+00:00)

LiteRT: applying dynamic INT8 quantization (int8 weights + FP32 activations)...


Applying Transformations to tensors:: 100%|██████████| 572/572 [00:00<00:00, 99007.18it/s]


Model name: safety_train_out/best_w8a32.tflite
Original model size: 10.06 MiB
Quantized model size: 2.79 MiB
Quantization Ratio: 0.28 (3.6x smaller)
Total time: 165.67 ms
LiteRT: export success ✅ 16.3s, saved as 'safety_train_out/best_w8a32.tflite' (2.8 MB)

Export complete (17.0s)
Results saved to /content/safety_train_out/best_w8a32.tflite
Predict:         yolo predict task=detect model=safety_train_out/best_w8a32.tflite imgsz=320 
Validate:        yolo val task=detect model=safety_train_out/best_w8a32.tflite imgsz=320 data=/content/helmet--1/data.yaml  
Visualize:       https://netron.app
w8a32_320       17s   2.79 MB


'w8a32_320.tflite'

---
## [STEP 5] LiteRT 정적 INT8 320 — 축 B를 끝까지

`quantize=8` : 가중치와 중간값 **모두** INT8. 가장 작고 빠르다.
대신 "값이 보통 어느 범위인지" 미리 재야 해서 **실제 사진이 필요**하다.

> Custom 학습의 이점 : **우리 데이터로 캘리브레이션**하므로 남의 모델보다 손실이 적다.
> 앞서 파이에서 쓴 `yolov8n_int8.tflite` 가 바로 이 방식이다.

In [10]:
if YAML_PATH:
    try:
        export_litert('int8_320', 320, quantize=8, data=str(YAML_PATH)) # 가중치 중간값 모두 8비트
    except Exception as e:
        print('정적 INT8 실패 — w8a32 로 대체해도 수업 진행에 지장 없다\n', e)
else:
    print('data.yaml 이 없어 건너뛴다')

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from 'safety_train_out/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (5.2 MB)
LiteRT: collecting INT8 calibration images from 'data=safety_train_out/data.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 960.3±277.9 MB/s, size: 61.2 KB)
val: Scanning /content/safety_train_out/safety_dataset/valid/labels... 113 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 113/113 1.2Kit/s 0.1s
val: New cache created: /content/safety_train_out/safety_dataset/valid/labels.cache
WARNING ⚠️ LiteRT: >300 images recommended for INT8 calibration, found 113 images.

LiteRT: starting export with litert_torch 0.9.3...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


...

(00:00) [START] Write Model to safety_train_out/best_int8.tflite

(00:00) [ DONE] Write Model to safety_train_out/best_int8.tflite (+00:00)

LiteRT: applying static quantization (int8 weights + int8 activations)...


/usr/local/lib/python3.12/dist-packages/ai_edge_litert/interpreter.py:480: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(
Applying Transformations to tensors:: 100%|██████████| 572/572 [00:00<00:00, 18686.22it/s]


Model name: safety_train_out/best_int8.tflite
Original model size: 10.06 MiB
Quantized model size: 2.88 MiB
Quantization Ratio: 0.29 (3.5x smaller)
Total time: 241.03 ms
LiteRT: export success ✅ 54.4s, saved as 'safety_train_out/best_int8.tflite' (2.9 MB)

Export complete (54.7s)
Results saved to /content/safety_train_out/best_int8.tflite
Predict:         yolo predict task=detect model=safety_train_out/best_int8.tflite imgsz=320 
Validate:        yolo val task=detect model=safety_train_out/best_int8.tflite imgsz=320 data=/content/helmet--1/data.yaml  
Visualize:       https://netron.app
int8_320        55s   2.88 MB


---
## [STEP 6] LiteRT FP32 640 — 축 C 비교용

해상도만 바꿔 320과 비교한다. **크기는 거의 같은데 속도는 4배 차이**가 난다.

In [11]:
export_litert('fp32_640', 640)#얘는 이미지 크기를 키움

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs

PyTorch: starting from 'safety_train_out/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 6, 8400) (5.2 MB)

LiteRT: starting export with litert_torch 0.9.3...


(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


...

(00:20) [ DONE] LiteRT-Torch Convert (+00:20)

(00:00) [START] Write Model to safety_train_out/best.tflite

(00:00) [ DONE] Write Model to safety_train_out/best.tflite (+00:00)

LiteRT: export success ✅ 21.0s, saved as 'safety_train_out/best.tflite' (10.1 MB)

Export complete (22.1s)
Results saved to /content/safety_train_out/best.tflite
Predict:         yolo predict task=detect model=safety_train_out/best.tflite imgsz=640 
Validate:        yolo val task=detect model=safety_train_out/best.tflite imgsz=640 data=/content/helmet--1/data.yaml  
Visualize:       https://netron.app
fp32_640        22s   10.14 MB


'fp32_640.tflite'

---
## [STEP 7] (선택) NCNN — 더 빠른 길 rassiberriy.py

Ultralytics 공식 RPi5 벤치마크 (입력 640, FP32)

| 포맷 | 추론 시간 |
| --- | --- |
| PyTorch `.pt` | 299 ms |
| **LiteRT** | **123 ms** |
| NCNN | **67 ms** |

NCNN은 ARM CPU 전용 엔진이라 한 번 더 빠르다. 시간이 남는 팀만 실행한다.
학교 네트워크에서 `pnnx` 다운로드가 막히면 실패하는데, **무시하고 넘어가도 된다.**

In [ ]:
try:
    t = time.time()
    YOLO(str(PT_PATH)).export(format='ncnn', imgsz=320)
    shutil.rmtree('ncnn_320', ignore_errors=True)
    shutil.move('best_ncnn_model', 'ncnn_320')
    MADE['ncnn_320'], LOG['ncnn_320'] = 'ncnn_320', time.time() - t
    print(f'ncnn_320 완료 {LOG["ncnn_320"]:.0f}s')
except Exception as e:
    print('NCNN 건너뜀 :', e)

Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (AMD EPYC 7B12)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 1.6 GFLOPs

PyTorch: starting from 'safety_train_out/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 6, 2100) (5.2 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 224ms
Prepared 1 package in 201ms
Installed 1 package in 1ms
 + ncnn==1.0.20260526

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx==20260526'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 11 packages in 249ms
Prepared 1 package in 427ms
Installed 1 package in 3ms
 + pnnx==20260526

requirements: AutoUpdate success 

---
## [STEP 8] 크기 비교 — 워크북 비교표 #1

In [12]:
import pandas as pd

def fsize_mb(p):
    p = str(p)
    if os.path.isdir(p):
        return sum(os.path.getsize(os.path.join(r, f))
                   for r, _, fs in os.walk(p) for f in fs) / 1024 / 1024
    return os.path.getsize(p) / 1024 / 1024

rows = [('PyTorch (원본)', fsize_mb(PT_PATH), 0)]
rows += [(tag, fsize_mb(p), LOG.get(tag, 0)) for tag, p in MADE.items()]

df = pd.DataFrame(rows, columns=['포맷', '크기(MB)', '변환(s)'])
df['원본대비'] = (df['크기(MB)'] / df['크기(MB)'].iloc[0]).round(2)
df

,포맷,크기(MB),변환(s),원본대비
0,PyTorch (원본),5.214075,0.000000,1.00
1,fp32_320,10.063241,21.478233,1.93
2,w8a32_320,2.787724,17.015385,0.53
3,int8_320,2.881955,54.748467,0.55
4,fp32_640,10.135341,22.136541,1.94


**생각해 볼 것**

1. `fp32_320` 과 `fp32_640` 의 크기가 왜 거의 같은가?

:그냥 pyton을 litert로 모델 학습 라이브러리를 바꾼것이기에 원본이랑 크기가 그대로이다
2. `int8_320` 은 `fp32_320` 의 몇 분의 1인가? 축 B에서 배운 숫자와 맞는가?

: 얘는 크기를 8로 줄여버렸기 때문에 8/32 =1/4 이다

---
## [STEP 9] 정확도 비교 ★ — 워크북 비교표 #2

**오늘 실습에서 가장 중요한 단계다.**

빨라졌다고 좋아할 일이 아니다. 안전 장치에서 최악의 실패는 **미착용자를 놓치는 것**이다.
양자화가 `No-helmet` 의 **Recall(재현율)** 을 얼마나 깎았는지 숫자로 확인한다.

> 포맷당 2~4분 걸린다. 돌려 놓고 강의를 듣는다.
> 출력에 나오는 **속도(ms)는 무시한다** — Colab CPU 시간은 파이와 아무 관계가 없다.

In [16]:

# 성능 비교
rows = []
targets = [('PyTorch (원본)', str(PT_PATH), 640)]
targets += [(t, p, 640 if '640' in t else 320)
            for t, p in MADE.items() if str(p).endswith('.tflite')]

for name, path, sz in targets:
    try:
        r = YOLO(path, task='detect').val(data=str(YAML_PATH), imgsz=sz,
                                          verbose=False, plots=False)
        rows.append({'모델': name,
                     'mAP50': round(r.box.map50, 3),
                     'mAP50-95': round(r.box.map, 3),
                     '위험 Recall': round(float(r.box.r[DANGER]), 3),
                     '위험 Precision': round(float(r.box.p[DANGER]), 3)})
        print('완료 :', name)
    except Exception as e:
        print('실패 :', name, e)

acc = pd.DataFrame(rows)
acc

Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO11n summary (fused): 101 layers, 2,582,542 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1542.6±147.2 MB/s, size: 52.9 KB)
val: Scanning /content/safety_train_out/safety_dataset/valid/labels.cache... 113 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 113/113 31.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 4.2s/it 33.3s
                   all        113        523      0.821      0.689      0.746      0.418
Speed: 6.6ms preprocess, 281.0ms inference, 0.0ms loss, 1.2ms postprocess per image
완료 : PyTorch (원본)
Ultralytics 8.4.118 🚀 Python-3.12.13 torch-2.11.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Loading fp32_320.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1160.0±154.0 MB/s, size: 61.7 KB)
val: Scann

,모델,mAP50,mAP50-95,위험 Recall,위험 Precision
0,PyTorch (원본),0.746,0.418,0.529,0.785
1,fp32_320,0.600,0.322,0.428,0.727
2,w8a32_320,0.599,0.331,0.425,0.725
3,int8_320,0.566,0.273,0.379,0.618
4,fp32_640,0.731,0.405,0.529,0.797


In [17]:
# 원본 대비 얼마나 떨어졌는가
if len(acc) > 1:
    base = acc.iloc[0]
    diff = acc.copy()
    for c in ['mAP50', 'mAP50-95', '위험 Recall', '위험 Precision']:
        diff[c] = (acc[c] - base[c]).round(3)
    display(diff.iloc[1:])
    print('위험 Recall 이 -0.05 라면 → 미착용자 100명 중 5명을 더 놓친다는 뜻이다.')
    #사이즈도 작으면서 원본 손실도 그렇게 많지 않은 애를 찾아야 한다

,모델,mAP50,mAP50-95,위험 Recall,위험 Precision
1,fp32_320,-0.146,-0.096,-0.101,-0.058
2,w8a32_320,-0.147,-0.087,-0.104,-0.060
3,int8_320,-0.180,-0.145,-0.150,-0.167
4,fp32_640,-0.015,-0.013,0.000,0.012


위험 Recall 이 -0.05 라면 → 미착용자 100명 중 5명을 더 놓친다는 뜻이다.


### 팀 토론 (5분)

1. 정적 INT8이 가장 빠르고 작다. 그런데 `No-helmet` Recall 이 얼마나 떨어졌나?
2. 그 손실을 속도 몇 ms 와 바꿀 만한가? 우리 학교 실습실이라면?
3. **우리 팀의 주력 모델은 무엇이고, 이유는 한 문장으로 무엇인가?**
원본에 비해서 크기도 작고 성능도 많이 떨어지지 않기에 따라서 w8a32를 사용해야 한다
> 정답은 없다. 다만 **이유를 대지 못한 선택**은 엔지니어링이 아니다.

---
## [STEP 10] RPi5 배포 패키지

모델 파일만 보내면 소용없다. **번호↔이름 사전**과 **실행 파일**이 같이 가야 한다.

In [18]:
TEAM = 'team01'         # ← 팀 이름
MAIN = 'w8a32_320'      # ← STEP 9 토론에서 정한 주력 모델
print('팀', TEAM, '/ 주력', MAIN)

팀 team01 / 주력 w8a32_320


In [19]:
DEPLOY = Path('rpi_deploy_safety')
shutil.rmtree(DEPLOY, ignore_errors=True)
DEPLOY.mkdir()

IMGSZ = {}
for tag, p in MADE.items():
    dst = DEPLOY / os.path.basename(p)
    shutil.copytree(p, dst) if os.path.isdir(p) else shutil.copy(p, dst)
    IMGSZ[os.path.basename(p)] = 640 if '640' in tag else 320
    print('포함 :', os.path.basename(p))

meta = {'team': TEAM,
        'main_model': os.path.basename(MADE[MAIN]),
        'classes': CLASSES,
        'danger_index': DANGER,
        'danger_name': CLASSES[DANGER],
        'imgsz': IMGSZ,
        'conf': 0.35,
        'serial': 'AI,<0=safe|1=danger>,<conf 0~100> @9600bps'}
(DEPLOY / 'safety_classes.json').write_text(
    json.dumps(meta, ensure_ascii=False, indent=2), encoding='utf-8')

# 테스트 이미지 한 장
img = next(Path('.').rglob('valid/images/*.jpg'), None)
if img:
    shutil.copy(img, DEPLOY / 'test_sample.jpg')
    print('테스트 이미지 :', img.name)

print(json.dumps(meta, ensure_ascii=False, indent=2))

포함 : fp32_320.tflite
포함 : w8a32_320.tflite
포함 : int8_320.tflite
포함 : fp32_640.tflite
테스트 이미지 : hard_hat_workers139_png_jpg.rf.1ad57c579d637b2b6dd8a41920748b59.jpg
{
  "team": "team01",
  "main_model": "w8a32_320.tflite",
  "classes": [
    "No-helmet",
    "helmet"
  ],
  "danger_index": 0,
  "danger_name": "No-helmet",
  "imgsz": {
    "fp32_320.tflite": 320,
    "w8a32_320.tflite": 320,
    "int8_320.tflite": 320,
    "fp32_640.tflite": 640
  },
  "conf": 0.35,
  "serial": "AI,<0=safe|1=danger>,<conf 0~100> @9600bps"
}


### 실행 파일 넣기

파이 실습에서는 `Interpreter` 로 텐서를 직접 다뤘다 — 원리를 알기 위해서였다.
이번엔 **`YOLO()` 한 줄**로 부른다. 우리가 만든 모델이라 전처리·후처리를 알아서 맞춰 준다.

> 아래 셀은 **읽고 이해할 코드가 아니라 그냥 실행할 셀**이다. 내용은 D9에서 뜯어본다.

In [ ]:
# 걍 하드웨어에서 넣고 돌릴 코드
SCRIPT = r"""#!/usr/bin/env python3
# detect_safety.py - RPi5 : 안전장비 미착용 감지 -> 아두이노 통보
#   python3 detect_safety.py                        # 기본
#   python3 detect_safety.py --no-show              # SSH 접속 시
#   python3 detect_safety.py --model int8_320.tflite    # 모델 비교
#   python3 detect_safety.py --port none            # 아두이노 없이
# 아두이노 신호 : AI,<0|1>,<신뢰도 0~100>\n  9600bps. 상태가 바뀔 때만 보낸다.

import argparse, json, os, sys, time
import cv2
from ultralytics import YOLO

HERE = os.path.dirname(os.path.abspath(__file__))
META = json.load(open(os.path.join(HERE, 'safety_classes.json'), encoding='utf-8'))

ap = argparse.ArgumentParser()
ap.add_argument('--model', default=META['main_model'])
ap.add_argument('--port',  default='/dev/ttyACM0')
ap.add_argument('--conf',  type=float, default=META.get('conf', 0.35))
ap.add_argument('--hold',  type=float, default=1.0)
ap.add_argument('--no-show', action='store_true')
a = ap.parse_args()

IMGSZ  = META['imgsz'].get(a.model, 320)
DANGER = META['danger_index']
print(f"model {a.model} | imgsz {IMGSZ} | danger {DANGER}:{META['danger_name']}")

ser = None
if a.port.lower() != 'none':
    try:
        import serial
        ser = serial.Serial(a.port, 9600, timeout=0.1)
        time.sleep(2)
        print('arduino:', a.port)
    except Exception as e:
        print('no arduino:', e)

def send(danger, conf):
    line = f"AI,{int(danger)},{int(conf * 100)}\n"
    if ser:
        ser.write(line.encode())
    print('->', line.strip())

model = YOLO(os.path.join(HERE, a.model), task='detect')
cap = cv2.VideoCapture(0)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
if not cap.isOpened():
    sys.exit('camera open failed')

state, since, sent = False, 0.0, None
tick, n, fps = time.time(), 0, 0.0
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        r = model(frame, imgsz=IMGSZ, conf=a.conf, verbose=False)[0]
        hits = [float(b.conf) for b in r.boxes if int(b.cls) == DANGER]
        now, best, t = len(hits) > 0, (max(hits) if hits else 0.0), time.time()

        if now != state:                      # hold 초 이상 지속돼야 상태 변경
            if since == 0.0:
                since = t
            elif t - since >= a.hold:
                state, since = now, 0.0
        else:
            since = 0.0

        if state != sent:
            send(state, best)
            sent = state

        n += 1
        if t - tick >= 1.0:
            fps, n, tick = n / (t - tick), 0, t

        if a.no_show:
            if n == 0:
                print(f'{"DANGER" if state else "SAFE  "}  {fps:4.1f} FPS')
        else:
            img = r.plot()
            cv2.putText(img, f'{"DANGER : no helmet" if state else "SAFE"}  {fps:.1f} FPS',
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8,
                        (0, 0, 255) if state else (0, 180, 0), 2)
            cv2.imshow('Smart Safety Station', img)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
finally:
    cap.release()
    cv2.destroyAllWindows()
    if ser:
        send(False, 0)
        ser.close()
    print('bye')
"""

(DEPLOY / 'detect_safety.py').write_text(SCRIPT, encoding='utf-8')
print('detect_safety.py 작성 완료')

detect_safety.py 작성 완료


In [ ]:
README = f"""[{TEAM}] 안전장비 탐지 배포 꾸러미
=========================================================
클래스 : {CLASSES}      <- 번호 순서. 절대 바꾸지 않는다.
위험   : {DANGER} = {CLASSES[DANGER]}
주력   : {os.path.basename(MADE[MAIN])}

RPi5 최초 설치 (10분쯤 걸린다. D8 전에 미리 해 둔다)
---------------------------------------------------------
  python3 -m venv --system-site-packages ~/edgeai/venv
  source ~/edgeai/venv/bin/activate
  pip install ultralytics pyserial

실행
---------------------------------------------------------
  cd ~/edgeai/models && unzip -o rpi_deploy_safety.zip -d safety && cd safety
  source ~/edgeai/venv/bin/activate
  python3 detect_safety.py                      # 화면 있을 때
  python3 detect_safety.py --no-show            # SSH 접속일 때
  python3 detect_safety.py --model fp32_640.tflite   # 모델 비교
  python3 detect_safety.py --port none          # 아두이노 없이

아두이노 신호 (USB 시리얼 9600bps)
---------------------------------------------------------
  AI,1,87  -> 미착용 감지, 신뢰도 87%
  AI,0,0   -> 정상 복귀
"""

(DEPLOY / 'README.txt').write_text(README, encoding='utf-8')
zip_name = shutil.make_archive('rpi_deploy_safety', 'zip', DEPLOY)
print(README)
print(f'-> {zip_name}  ({os.path.getsize(zip_name)/1024/1024:.1f} MB)')
print('\nRPi5 전송 :')
print('  scp rpi_deploy_safety.zip admin@<RPi-IP>:~/edgeai/models/')

[team01] 안전장비 탐지 배포 꾸러미
클래스 : ['No-helmet', 'helmet']      <- 번호 순서. 절대 바꾸지 않는다.
위험   : 0 = No-helmet
주력   : w8a32_320.tflite

RPi5 최초 설치 (10분쯤 걸린다. D8 전에 미리 해 둔다)
---------------------------------------------------------
  python3 -m venv --system-site-packages ~/edgeai/venv
  source ~/edgeai/venv/bin/activate
  pip install ultralytics pyserial

실행
---------------------------------------------------------
  cd ~/edgeai/models && unzip -o rpi_deploy_safety.zip -d safety && cd safety
  source ~/edgeai/venv/bin/activate
  python3 detect_safety.py                      # 화면 있을 때
  python3 detect_safety.py --no-show            # SSH 접속일 때
  python3 detect_safety.py --model fp32_640.tflite   # 모델 비교
  python3 detect_safety.py --port none          # 아두이노 없이

아두이노 신호 (USB 시리얼 9600bps)
---------------------------------------------------------
  AI,1,87  -> 미착용 감지, 신뢰도 87%
  AI,0,0   -> 정상 복귀

-> /content/rpi_deploy_safety.zip  (22.0 MB)

RPi5 전송 :
  scp rpi_deploy_safety.zip admin@<RPi-IP>:~/ed

In [ ]:
from google.colab import files
files.download('rpi_deploy_safety.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## [STEP 11] 파이에서 채울 표 (D8·D9 예고)

Colab 속도는 **한 번도 쓰지 않았다.** 의도한 것이다.
`detect_safety.py` 화면 왼쪽 위 FPS 로 이 표를 채운다.

| 모델 | RPi5 FPS | mAP50 (STEP 9) | 위험 Recall (STEP 9) |
| --- | --- | --- | --- |
| `fp32_640.tflite` | | | |
| `fp32_320.tflite` | | | |
| `w8a32_320.tflite` | | | |
| `int8_320.tflite` | | | |
| `ncnn_320` (선택) | | | |

속도와 정확도를 양손에 놓고 저울질하는 것 — 그것이 엣지 AI 엔지니어의 일이다.

---
## 체크포인트

- [ ] FP32 / w8a32 / INT8 / 640 네 가지 `.tflite` 생성
- [ ] 정적 INT8은 **우리 데이터**로 캘리브레이션됨
- [ ] STEP 8 크기표, STEP 9 정확도표를 워크북에 옮겨 적음
- [ ] 주력 모델을 **이유와 함께** 결정
- [ ] `rpi_deploy_safety.zip` 생성 및 두 곳 백업
- [ ] `safety_classes.json` 의 0번이 무엇인지 팀원 전원이 답할 수 있음

---
> **안전 관점** : 경량화는 속도를 얻고 정확도를 조금 내주는 거래다.
> 0.1초 빨라지는 것보다 **미착용자 한 명을 놓치지 않는 것**이 중요하다.
> 그 판단이 `--conf` 값 한 줄로 바뀐다. 그 한 줄이 엔지니어의 책임이다.
